# Chapter 4: Exploratory Data Analysis (EDA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aeleraqi/Python-for-Journalists/blob/main/notebooks/Chapter_04_Notebook.ipynb)

Companion notebook to *The Code Behind the Story: Data Skills with Python for Journalists*, second edition, by Amr Eleraqi. It holds every code cell of the chapter, in the book's order; the book explains each one. Run the cells from top to bottom.

## 4.1 Looking before you write

### 4.1.1 The data for this chapter

In [ ]:
repo = ("https://raw.githubusercontent.com/"
        "aeleraqi/Python-for-Journalists/main")
!wget -q {repo}/data/Housing.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

housing = pd.read_csv("Housing.csv")
housing.shape

In [ ]:
housing.info()

In [ ]:
housing.head(3).T

In [ ]:
housing.sample(5, random_state=7).T

## 4.2 Data manipulation techniques

### 4.2.1 Selecting columns

In [ ]:
housing[["price", "area", "bedrooms", "airconditioning"]].head()

### 4.2.2 Filtering rows

In [ ]:
with_ac = housing[housing["airconditioning"] == "yes"]
len(with_ac)

In [ ]:
big_with_ac = housing[(housing["airconditioning"] == "yes") &
                      (housing["bedrooms"] >= 4)]
big_with_ac[["price", "area", "bedrooms", "stories"]].head()

In [ ]:
furnished = housing[housing["furnishingstatus"].isin(
    ["furnished", "semi-furnished"])]
mid_size = housing[housing["area"].between(4000, 6000)]
print(len(furnished), len(mid_size))

In [ ]:
# This cell stops with an error on purpose: read the message.
housing[housing["airconditioning"] == "yes" & housing["bedrooms"] >= 4]

### 4.2.3 Sorting

In [ ]:
by_size = housing.sort_values(["area", "price"], ascending=[False, True])
by_size[["area", "price", "bedrooms", "prefarea"]].head()

### 4.2.4 Adding columns

In [ ]:
housing["price_per_area"] = housing["price"] / housing["area"]
housing[["price", "area", "price_per_area"]].head(3)

In [ ]:
housing["ac"] = housing["airconditioning"].map({"yes": 1, "no": 0})
housing["ac"].value_counts()

In [ ]:
def size_label(area):
    if area > 4600:
        return "large"
    else:
        return "small"

housing["size"] = housing["area"].apply(size_label)
housing["size"].value_counts()

In [ ]:
housing = housing.drop(columns=["price_per_area", "ac", "size"])
housing.shape

### 4.2.5 Grouping and aggregating

In [ ]:
housing.groupby("furnishingstatus")["price"].mean()

In [ ]:
pd.set_option("display.float_format", "{:,.2f}".format)
housing.groupby("furnishingstatus")["price"].mean()

In [ ]:
by_furnishing = housing.groupby("furnishingstatus").agg(
    houses=("price", "size"),
    mean_price=("price", "mean"),
    median_price=("price", "median"),
    median_area=("area", "median"),
)
by_furnishing

### 4.2.6 Pivot tables

In [ ]:
housing.pivot_table(values="price", index="furnishingstatus",
                    columns="airconditioning", aggfunc="median")

In [ ]:
housing.pivot_table(values="price", index="furnishingstatus",
                    columns="airconditioning", aggfunc="count",
                    margins=True, fill_value=0)

### 4.2.7 Reshaping from wide to long with melt()

In [ ]:
yes_no_columns = ["mainroad", "guestroom", "basement",
                  "hotwaterheating", "airconditioning", "prefarea"]

features = housing.melt(id_vars="price", value_vars=yes_no_columns,
                        var_name="feature", value_name="has_it")
print(features.shape)
features.head()

In [ ]:
feature_table = features.pivot_table(values="price", index="feature",
                                     columns="has_it", aggfunc="median")
feature_table["houses_with_it"] = features[
    features["has_it"] == "yes"].groupby("feature").size()
feature_table.sort_values("houses_with_it")

## 4.3 Univariate analysis: one column at a time

### 4.3.1 Summary statistics

In [ ]:
housing.describe()

In [ ]:
housing.describe(exclude="number").T

### 4.3.2 The center: mean, median and mode

In [ ]:
mean_price = housing["price"].mean()
median_price = housing["price"].median()
print(f"Mean price:   {mean_price:,.2f}")
print(f"Median price: {median_price:,.2f}")

In [ ]:
print(round(housing["price"].skew(), 2))

In [ ]:
housing["price"].mode()

### 4.3.3 The spread: range, IQR and standard deviation

In [ ]:
prices = housing["price"]
q1 = prices.quantile(0.25)
q3 = prices.quantile(0.75)

print(f"Range:              {prices.max() - prices.min():,.0f}")
print(f"IQR:                {q3 - q1:,.0f}")
print(f"Standard deviation: {prices.std():,.0f}")
print(f"Variance:           {prices.var():,.0f}")

### 4.3.4 The shape: histograms

In [ ]:
plot_data = housing.assign(price_m=housing["price"] / 1_000_000)

In [ ]:
fig, ax = plt.subplots()
ax.hist(plot_data["price_m"], bins=30)
ax.axvline(plot_data["price_m"].mean(), color="C1", label="mean")
ax.axvline(plot_data["price_m"].median(), color="C2", linestyle="--",
           label="median")
ax.set_title("Distribution of house prices")
ax.set_xlabel("Price in millions (currency not documented)")
ax.set_ylabel("Number of houses")
ax.legend()
plt.show()

### 4.3.5 Box plots and outliers

In [ ]:
ax = plot_data["price_m"].plot.box()
ax.set_title("House prices")
ax.set_xticklabels(["All 545 houses"])
ax.set_ylabel("Price in millions (currency not documented)")
plt.show()

In [ ]:
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
print(f"Limits: {lower_limit:,.0f} to {upper_limit:,.0f}")

high = housing[housing["price"] > upper_limit]
print(len(high), "houses above the upper limit")

### 4.3.6 Counting categories

In [ ]:
housing["furnishingstatus"].value_counts()

In [ ]:
housing["furnishingstatus"].value_counts(normalize=True)

In [ ]:
counts = housing["furnishingstatus"].value_counts()
ax = counts.plot.bar(rot=0)
ax.set_title("Houses by furnishing status")
ax.set_xlabel("Furnishing status")
ax.set_ylabel("Number of houses")
plt.show()

## 4.4 Bivariate and multivariate analysis

### 4.4.1 Correlation

In [ ]:
r = housing["price"].corr(housing["area"])
print(round(r, 3))

In [ ]:
rho = housing["price"].corr(housing["area"], method="spearman")
print(round(rho, 3))

### 4.4.2 Scatter plots

In [ ]:
ax = sns.scatterplot(data=plot_data, x="area", y="price_m",
                     hue="airconditioning", hue_order=["no", "yes"])
ax.set_title("Price and area")
ax.set_xlabel("Area (unit not documented)")
ax.set_ylabel("Price in millions (currency not documented)")
ax.legend(title="Air conditioning")
plt.show()

### 4.4.3 The correlation matrix and its heatmap

In [ ]:
# This cell stops with an error on purpose: read the message.
housing.corr()

In [ ]:
numeric = housing.drop(columns="furnishingstatus")
for column in yes_no_columns:
    numeric[column] = numeric[column].map({"yes": 1, "no": 0})

correlations = numeric.corr()
correlations["price"].sort_values(ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.8))
sns.heatmap(correlations, annot=True, fmt=".2f", vmin=-1, vmax=1,
            cmap="coolwarm", annot_kws={"size": 6.5},
            cbar_kws={"label": "Pearson's r"}, ax=ax)
ax.set_title("Correlations between the columns")
plt.show()

### 4.4.4 Cross-tabulation

In [ ]:
pd.crosstab(housing["furnishingstatus"], housing["airconditioning"])

In [ ]:
pd.crosstab(housing["furnishingstatus"], housing["airconditioning"],
            normalize="index")

### 4.4.5 Comparing groups

In [ ]:
housing.groupby("airconditioning")["price"].agg(["size", "median"])

In [ ]:
ax = plot_data.boxplot(column="price_m", by="airconditioning")
plt.suptitle("")
ax.set_title("Price by air conditioning")
ax.set_xlabel("Air conditioning")
ax.set_ylabel("Price in millions (currency not documented)")
plt.show()

## 4.5 Practical tips for EDA

## 4.6 Case study: does air conditioning go with higher prices?

### Step 1: State the question

### Step 2: Check whether the groups differ in size

In [ ]:
housing.groupby("airconditioning")["area"].agg(["size", "median"])

### Step 3: Compare houses of similar size

In [ ]:
plot_data["size_band"] = pd.qcut(
    plot_data["area"], 4,
    labels=["Smallest", "Small", "Large", "Largest"])

plot_data.pivot_table(values="price_m", index="size_band",
                      columns="airconditioning",
                      aggfunc=["size", "median"], observed=True)

In [ ]:
ax = sns.barplot(data=plot_data, x="size_band", y="price_m",
                 hue="airconditioning", hue_order=["no", "yes"],
                 estimator="median", errorbar=None)
ax.set_title("Median price by size band and air conditioning")
ax.set_xlabel("Size band (quarters of the houses, by area)")
ax.set_ylabel("Median price in millions (currency not documented)")
ax.legend(title="Air conditioning")
plt.show()

### Step 4: Say what the evidence supports, and what it does not

## 4.7 Bonus: an automated first look with data-profiling

In [ ]:
!pip install fg-data-profiling

In [ ]:
from data_profiling import ProfileReport

In [ ]:
profile = ProfileReport(housing, title="Housing prices: automated profile")
profile.to_file("housing_report.html")

In [ ]:
profile.to_notebook_iframe()

## 4.8 Hands-on exercises

### Exercise 1: Price per unit of area by furnishing status

In [ ]:
housing["price_per_area"] = housing["price"] / housing["area"]
housing.groupby("furnishingstatus").agg(
    houses=("price_per_area", "size"),
    median_per_area=("price_per_area", "median"),
)

### Exercise 2: What the most expensive houses have in common

In [ ]:
housing["expensive"] = housing["price"] > upper_limit
housing.groupby("expensive")[["airconditioning", "prefarea"]].agg(
    lambda column: (column == "yes").mean())

### Exercise 3: Air conditioning within each furnishing group

In [ ]:
housing.pivot_table(values="price", index="furnishingstatus",
                    columns="airconditioning", aggfunc="median")

In [ ]:
housing.pivot_table(values="price", index="furnishingstatus",
                    columns="airconditioning", aggfunc="count")

### Exercise 4: Trace the data to its source

In [ ]:
url = ("https://raw.githubusercontent.com/vincentarelbundock/"
       "Rdatasets/master/csv/Ecdat/Housing.csv")
windsor = pd.read_csv(url)
print(windsor.shape)
windsor.columns

In [ ]:
comparison = pd.DataFrame({
    "kaggle_price_divided_by_70": (housing["price"] / 70).describe(),
    "windsor_price": windsor["price"].describe(),
})
comparison

In [ ]:
kaggle_rows = housing[yes_no_columns + ["price", "area", "bedrooms",
                                         "bathrooms", "stories", "parking"]]
windsor_rows = windsor.rename(columns={
    "lotsize": "area", "bathrms": "bathrooms", "driveway": "mainroad",
    "recroom": "guestroom", "fullbase": "basement",
    "gashw": "hotwaterheating", "airco": "airconditioning",
    "garagepl": "parking"}).drop(columns="rownames")
windsor_rows["price"] = windsor_rows["price"] * 70

matched = kaggle_rows.merge(windsor_rows.drop_duplicates(), how="inner")
print(len(matched), "of", len(kaggle_rows), "rows matched")

### Exercise 5: Price by number of floors

In [ ]:
ax = plot_data.boxplot(column="price_m", by="stories")
plt.suptitle("")
ax.set_title("Price by number of stories")
ax.set_xlabel("Stories")
ax.set_ylabel("Price in millions (currency not documented)")
plt.show()

print(housing["stories"].value_counts().sort_index())

## Solutions to the "Your turn" exercises

In [ ]:
housing = pd.read_csv("Housing.csv")
plot_data = housing.assign(price_m=housing["price"] / 1_000_000)

### Section 4.1: Looking before you write

In [ ]:
# 1
housing.tail().T

In [ ]:
# 2
housing.columns

In [ ]:
# 4
print(len(housing[housing["bedrooms"] == 3]))

# 3
housing.sample(3, random_state=1).T

### Section 4.2: Data manipulation techniques

In [ ]:
# 1
housing[["price", "bedrooms", "bathrooms"]].head(3)

In [ ]:
# 2
no_ac_big = housing[(housing["bedrooms"] >= 4) &
                    (housing["airconditioning"] == "no")]
print(len(no_ac_big))

# 3
housing.nlargest(10, "area")[["area", "price"]]

In [ ]:
# 4
housing.groupby("stories").agg(
    houses=("price", "size"),
    median_price=("price", "median"),
)

In [ ]:
# 5
housing.pivot_table(values="price", index="stories", columns="prefarea",
                    aggfunc="count")

### Section 4.3: Univariate analysis

In [ ]:
# 1
print(f"Mean area:   {housing['area'].mean():,.2f}")
print(f"Median area: {housing['area'].median():,.2f}")

# 2
area_q1 = housing["area"].quantile(0.25)
area_q3 = housing["area"].quantile(0.75)
area_iqr = area_q3 - area_q1
area_limit = area_q3 + 1.5 * area_iqr
print(f"IQR:         {area_iqr:,.0f}")
print(f"Upper limit: {area_limit:,.0f}")
print((housing["area"] > area_limit).sum(), "houses above it")

In [ ]:
# 3
ax = housing["area"].plot.hist(bins=20)
ax.set_title("Distribution of lot areas")
ax.set_xlabel("Area (unit not documented)")
ax.set_ylabel("Number of houses")
plt.show()

In [ ]:
# 4
print(housing["prefarea"].value_counts(normalize=True))

# 5
print(housing["stories"].value_counts().sort_index())

### Section 4.4: Bivariate and multivariate analysis

In [ ]:
# 1
print(round(housing["price"].corr(housing["bathrooms"]), 3))
print(round(housing["price"].corr(housing["bathrooms"],
                                  method="spearman"), 3))

In [ ]:
# 2
ax = sns.scatterplot(data=plot_data, x="area", y="price_m", hue="prefarea",
                     hue_order=["no", "yes"])
ax.set_title("Price and area, by preferred area")
ax.set_xlabel("Area (unit not documented)")
ax.set_ylabel("Price in millions (currency not documented)")
ax.legend(title="Preferred area")
plt.show()

In [ ]:
# 3
pd.crosstab(housing["prefarea"], housing["airconditioning"],
            normalize="index")

In [ ]:
# 4
housing.groupby("basement")["price"].agg(["size", "median"])